In [ ]:
!pip install numpy pandas statsmodels matplotlib yfinance

"""Quant Finance: pares Visa (V) e Mastercard (MA), versão corrigida.

Requer: numpy, pandas, statsmodels, matplotlib e yfinance.
"""
import os
import warnings

import numpy as np
import pandas as pd
import statsmodels.api as sm
from statsmodels.tsa.stattools import adfuller, coint, kpss
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore")
plt.style.use("dark_background")
os.makedirs("graficos_pairs_corrigido", exist_ok=True)

#Passo 1: parâmetros
FONTE = os.environ.get("FONTE", "yfinance")     # "yfinance" ou "sintetico" (teste sem internet)
A, B, MKT = "V", "MA", "SPY"                    # ativo A (comprado quando barato), ativo B, mercado
INICIO, FIM = "2016-01-01", "2026-01-01"
F = 250                 # janela de formação (pregões): estima beta, média e desvio do spread
W = 63                  # janela de operação (pregões) antes de reestimar tudo
H = 5                   # horizonte do retorno relativo da regra incondicional
K_IN, K_OUT = 2.0, 0.5  # entra com |z| > K_IN e sai com |z| < K_OUT
C_LEG = 0.0005          # custo por perna a cada troca (0,05%); cada troca negocia 1 + beta unidades
ALUGUEL = 0.0           # taxa anual de aluguel da ponta vendida (ASSUMIDA: confira com sua corretora)
RF_ANUAL = 0.0          # taxa livre de risco para os benchmarks comprar e segurar (AJUSTE)
RECORTES = {"10 anos": None, "últimos 5 anos": "2021-01-01"}
HORIZONTES = [1, 2, 5, 10, 21, 63, 126, 252]
N_BOOT, BLOCO_BOOT = 5000, 20
rng = np.random.default_rng(42)

#Passo 2: dados
def carregar():
    if FONTE == "yfinance":
        import yfinance as yf
        px = yf.download([A, B, MKT], start=INICIO, end=FIM, auto_adjust=True,
                         progress=False)["Close"].dropna()
        return px[[A, B, MKT]]
    g = np.random.default_rng(1)                 # sintético: spread AR(1) com phi conhecido
    n = 2500
    r_m = g.normal(0.0004, 0.010, n)
    lnB = np.log(100) + np.cumsum(r_m + g.normal(0, 0.008, n))
    s = np.zeros(n)
    for t in range(1, n):
        s[t] = 0.99 * s[t - 1] + g.normal(0, 0.004)
    lnA = np.log(60) + 0.82 * (lnB - lnB[0]) + s
    idx = pd.bdate_range(INICIO, periods=n)
    return pd.DataFrame({A: np.exp(lnA), B: np.exp(lnB), MKT: 100 * np.exp(np.cumsum(r_m))}, index=idx)

precos = carregar()
ln = np.log(precos)
ret = ln.diff().dropna()
n = len(precos)
lnA, lnB = ln[A].values, ln[B].values
rA, rB, rM = (np.r_[np.nan, ret[c].values] for c in (A, B, MKT))   # retorno do dia t (t = 0 não existe)
datas = precos.index
print(f"Amostra: {datas[0].date()} a {datas[-1].date()} ({n} pregões) | fonte: {FONTE}")

beta_cheio = np.cov(ret[A], ret[B])[0, 1] / ret[B].var()
print(f"Beta na amostra inteira (só diagnóstico, NÃO usado para operar): {beta_cheio:.4f}")

#Gráfico 1: preços normalizados e spread em nível
s_cheio = ln[A] - beta_cheio * ln[B]
fig, (a1, a2) = plt.subplots(2, 1, figsize=(8, 6), sharex=True)
(precos[[A, B]] / precos[[A, B]].iloc[0] * 100).plot(ax=a1, color=["#FFDF00", "#4FC3F7"])
a1.set_ylabel("Preço normalizado (início = 100)")
s_cheio.plot(ax=a2, color="white", lw=1)
a2.axhline(s_cheio.mean(), color="gray", ls="--")
for k in (-2, 2):
    a2.axhline(s_cheio.mean() + k * s_cheio.std(), color="#4FC3F7", ls=":")
a2.set_ylabel("Spread em nível (amostra inteira)")
plt.savefig("graficos_pairs_corrigido/grafico1_precos_spread.png", dpi=200,
            bbox_inches="tight", facecolor="black")
plt.show()

#Passo 3: estacionariedade, cointegração e meia-vida (com intervalo)
def meia_vida(phi):
    return np.log(0.5) / np.log(phi) if 0 < phi < 1 else np.inf

def ar1(s):
    y, x = np.asarray(s)[1:], np.asarray(s)[:-1]
    m = sm.OLS(y, sm.add_constant(x)).fit()
    return m.params[1], m.bse[1]

phi, se = ar1(s_cheio)
print("\nAmostra inteira, spread com beta fixo (diagnóstico):")
print(f"  ADF (H0: raiz unitária)      p = {adfuller(s_cheio, regression='c', autolag='AIC')[1]:.3f}")
print(f"  Engle-Granger (H0: sem coint.) p = {coint(ln[A], ln[B], trend='c', autolag='aic')[1]:.3f}")
print(f"  KPSS (H0: estacionária)      p = {kpss(s_cheio, regression='c', nlags='auto')[1]:.3f}")
print(f"  phi = {phi:.4f} (EP {se:.4f}, t de Dickey-Fuller ~ {(phi - 1) / se:.2f})")
print(f"  meia-vida = {meia_vida(phi):.1f} pregões | IC95% aprox. "
      f"[{meia_vida(phi - 1.96 * se):.1f}, {meia_vida(phi + 1.96 * se):.1f}] (inf = sem reversão)")

#Passo 4: janelas walk-forward (formação, depois operação)
blocos = [(t0, min(t0 + W, n)) for t0 in range(F, n - 2, W)]
beta_arr = np.full(n, np.nan)       # beta vigente em cada dia operado
d_ext = np.zeros(n)                 # posição da regra por extremos, definida no fechamento de t
d_inc = np.zeros(n)                 # posição da regra incondicional
sv = pd.Series(rA).rolling(H).sum().values
sb = pd.Series(rB).rolling(H).sum().values
janelas = []
for t0, t1 in blocos:
    f = slice(t0 - F, t0)
    b = np.cov(rA[t0 - F + 1:t0], rB[t0 - F + 1:t0])[0, 1] / np.var(rB[t0 - F + 1:t0], ddof=1)
    sf = lnA[f] - b * lnB[f]
    mu, sd = sf.mean(), sf.std(ddof=1)
    ph, _ = ar1(sf)
    janelas.append(dict(beta=b, phi=ph, hl=meia_vida(ph), adf=adfuller(sf, autolag="AIC")[1],
                        eg=coint(lnA[f], lnB[f], autolag="aic")[1]))
    pos = 0
    for t in range(t0, t1):
        z = ((lnA[t] - b * lnB[t]) - mu) / sd
        if z > K_IN:
            pos = -1                 # A cara em relação a B: vende A, compra B
        elif z < -K_IN:
            pos = 1
        elif abs(z) < K_OUT:
            pos = 0
        d_ext[t] = pos
        beta_arr[t] = b
        d_inc[t] = -np.sign(sv[t] - b * sb[t])
    d_ext[t1 - 1] = 0                # fecha todas as posições no fim da janela
    d_inc[t1 - 1] = 0

jan = pd.DataFrame(janelas)
print(f"\n{len(blocos)} janelas de formação de {F} pregões, operando {W} pregões cada")
print(f"  beta por janela: média {jan.beta.mean():.3f} | mín {jan.beta.min():.3f} | máx {jan.beta.max():.3f}")
print(f"  janelas com ADF p<0,05: {(jan.adf < 0.05).mean():.0%} | "
      f"com Engle-Granger p<0,05: {(jan.eg < 0.05).mean():.0%}")
fin = jan.hl[np.isfinite(jan.hl)]
print(f"  meia-vida por janela: mediana {fin.median():.0f} pregões "
      f"(quartis {fin.quantile(.25):.0f} a {fin.quantile(.75):.0f}); sem reversão em {(~np.isfinite(jan.hl)).mean():.0%}")

#Gráfico 2: o beta é estável? (janela móvel de 250 pregões)
beta_movel = (pd.Series(rA).rolling(250).cov(pd.Series(rB)) / pd.Series(rB).rolling(250).var())
plt.figure(figsize=(7, 4))
plt.plot(datas, beta_movel.values, color="#FFDF00", label="beta móvel (250 pregões)")
plt.axhline(beta_cheio, color="white", ls="--", label=f"beta da amostra inteira = {beta_cheio:.2f}")
plt.ylabel("Beta de cobertura")
plt.legend()
plt.savefig("graficos_pairs_corrigido/grafico2_beta_movel.png", dpi=200,
            bbox_inches="tight", facecolor="black")
plt.show()

#Passo 5: regressão preditiva com Newey-West, em uma grade de horizontes
# retorno hedgeado diário com beta móvel defasado (só informação do passado)
hr = pd.Series(rA) - beta_movel.shift(1) * pd.Series(rB)
linhas = []
for h in HORIZONTES:
    X = hr.rolling(h).sum()
    df = pd.concat([X.shift(-h), X], axis=1).dropna()
    df.columns = ["Y", "X"]
    m = sm.OLS(df.Y, sm.add_constant(df.X)).fit(cov_type="HAC", cov_kwds={"maxlags": 2 * h})
    dn = df.iloc[::h]
    mn = sm.OLS(dn.Y, sm.add_constant(dn.X)).fit()
    linhas.append(dict(h=h, n=len(df), b=m.params["X"], ep=m.bse["X"], t=m.tvalues["X"],
                       p=m.pvalues["X"], b_ns=mn.params["X"], t_ns=mn.tvalues["X"], n_ns=len(dn)))
grade = pd.DataFrame(linhas)
print("\nY = R(t, t+h) contra X = R(t-h, t). Newey-West com 2h defasagens; '_ns' = sem sobreposição")
print(grade.round(4).to_string(index=False))

#Gráfico 3: b por horizonte com intervalo de 95%
plt.figure(figsize=(7, 4))
x = np.arange(len(grade))
plt.errorbar(x, grade.b, yerr=1.96 * grade.ep, fmt="o", color="#FFDF00", capsize=4,
             label="b com IC 95% (Newey-West)")
plt.plot(x, grade.b_ns, "s", color="#4FC3F7", label="b sem sobreposição")
plt.axhline(0, color="gray", ls="--")
plt.xticks(x, [f"{h}d" for h in grade.h])
plt.xlabel("Horizonte h = k (dias)")
plt.ylabel("Coeficiente b (negativo = reversão)")
plt.legend()
plt.savefig("graficos_pairs_corrigido/grafico3_b_por_horizonte.png", dpi=200,
            bbox_inches="tight", facecolor="black")
plt.show()

#Passo 6: retornos líquidos (duas pernas, custos e aluguel) e métricas
def liquidos(d, c_leg=C_LEG, aluguel=ALUGUEL):
    dp = np.r_[0, d[:-1]]                       # posição que vale durante o dia t
    dp2 = np.r_[0, 0, d[:-2]]
    bp = np.nan_to_num(np.r_[np.nan, beta_arr[:-1]])
    bruto = dp * (rA - bp * rB)
    custo = c_leg * np.abs(dp - dp2) * (1 + bp)         # cada troca negocia as duas pernas
    curta = np.where(dp > 0, bp, np.where(dp < 0, 1.0, 0.0))
    return np.nan_to_num(bruto - custo - aluguel / 252 * curta), dp

def operacoes(r, dp):
    pnl, dur, ativa, acc, dias = [], [], False, 0.0, 0
    for t in range(len(r)):
        if dp[t] != 0:
            if not ativa or dp[t] != dp[t - 1]:
                if ativa:
                    pnl.append(acc); dur.append(dias)
                acc, dias, ativa = 0.0, 0, True
            acc += r[t]; dias += 1
        elif ativa:                                      # dia da saída (custo de saída)
            acc += r[t]; pnl.append(acc); dur.append(dias)
            acc, dias, ativa = 0.0, 0, False
    if ativa:
        pnl.append(acc); dur.append(dias)
    return np.array(pnl), np.array(dur)

def resumo(r, dp):
    anos = len(r) / 252
    sr = r.mean() / r.std(ddof=1) * np.sqrt(252)
    riq = np.cumprod(1 + r)
    pnl, dur = operacoes(r, dp)
    return dict(ret_aa=r.mean() * 252, ret_comp=(1 + r.mean()) ** 252 - 1, vol_aa=r.std(ddof=1) * np.sqrt(252),
                sharpe=sr, t_sharpe=sr * np.sqrt(anos), mdd=(riq / np.maximum.accumulate(riq) - 1).min(),
                tempo_pos=(dp != 0).mean(), n_op=len(pnl), op_ano=len(pnl) / anos,
                acerto=(pnl > 0).mean() if len(pnl) else np.nan, dur_med=dur.mean() if len(dur) else np.nan)

usavel = ~np.isnan(beta_arr)
primeiro = np.argmax(usavel) + 1
r_ext, dp_ext = liquidos(d_ext)
r_inc, dp_inc = liquidos(d_inc)
tabela = {}
for nome_rec, inicio_rec in RECORTES.items():
    sel = np.arange(n) >= primeiro
    if inicio_rec:
        sel &= np.asarray(datas >= pd.Timestamp(inicio_rec))
    for nome_reg, r, dp in (("extremos |z|>2", r_ext, dp_ext), ("incondicional (sinal de R)", r_inc, dp_inc)):
        tabela[(nome_rec, nome_reg)] = resumo(r[sel], dp[sel])
res = pd.DataFrame(tabela).T
print("\nMesma amostra, mesma base (retorno = média diária x 252, por unidade de A; custos nas duas pernas)")
print(res.round(3).to_string())

#Gráfico 4: riqueza acumulada das duas regras na mesma amostra
plt.figure(figsize=(7, 4))
sel10 = np.arange(n) >= primeiro
plt.plot(datas[sel10], np.cumprod(1 + r_ext[sel10]), color="#FFDF00", lw=2, label="extremos |z|>2")
plt.plot(datas[sel10], np.cumprod(1 + r_inc[sel10]), color="#4FC3F7", label="incondicional")
plt.axhline(1, color="gray", ls="--")
plt.ylabel("Riqueza (início = 1)")
plt.legend()
plt.savefig("graficos_pairs_corrigido/grafico4_riqueza.png", dpi=200,
            bbox_inches="tight", facecolor="black")
plt.show()

#Passo 7: o Sharpe é distinguível de zero? (bootstrap estacionário em blocos)
def sharpe_boot(r, g):
    m = len(r)
    idx = np.empty((N_BOOT, m), dtype=np.int32)
    idx[:, 0] = g.integers(0, m, N_BOOT)
    for t in range(1, m):
        novo = g.random(N_BOOT) < 1 / BLOCO_BOOT
        idx[:, t] = np.where(novo, g.integers(0, m, N_BOOT), (idx[:, t - 1] + 1) % m)
    amostra = r[idx]
    return amostra.mean(1) / amostra.std(1, ddof=1) * np.sqrt(252)

boots = {}
print("\nBootstrap do Sharpe (blocos médios de 20 dias):")
for nome_reg, r in (("extremos |z|>2", r_ext), ("incondicional (sinal de R)", r_inc)):
    sb_ = sharpe_boot(r[sel10], np.random.default_rng(7))
    boots[nome_reg] = sb_
    print(f"  {nome_reg:<28} IC95% [{np.percentile(sb_, 2.5):.2f}, {np.percentile(sb_, 97.5):.2f}] | "
          f"P(Sharpe <= 0) = {(sb_ <= 0).mean():.1%}")

#Gráfico 5: distribuição bootstrap do Sharpe
plt.figure(figsize=(7, 4))
for (nome_reg, v), cor in zip(boots.items(), ["#FFDF00", "#4FC3F7"]):
    plt.hist(v, bins=60, histtype="step", lw=2, color=cor, density=True, label=nome_reg)
plt.axvline(0, color="white", ls="--")
plt.xlabel("Sharpe anualizado (bootstrap)")
plt.yticks([])
plt.legend()
plt.savefig("graficos_pairs_corrigido/grafico5_sharpe_bootstrap.png", dpi=200,
            bbox_inches="tight", facecolor="black")
plt.show()

#Passo 8: sensibilidade aos custos
custos = [0.0, 0.0002, 0.0005, 0.001, 0.002]
sens = {nome: [] for nome in ("extremos |z|>2", "incondicional (sinal de R)")}
for c in custos:
    for nome, d in (("extremos |z|>2", d_ext), ("incondicional (sinal de R)", d_inc)):
        r_c, _ = liquidos(d, c_leg=c)
        sens[nome].append(r_c[sel10].mean() / r_c[sel10].std(ddof=1) * np.sqrt(252))
print("\nSharpe conforme o custo por perna (extremos | incondicional):")
for i, c in enumerate(custos):
    print(f"  {c:.2%}: {sens['extremos |z|>2'][i]:>6.2f} | {sens['incondicional (sinal de R)'][i]:>6.2f}")

#Gráfico 6: o Sharpe sobrevive aos custos?
plt.figure(figsize=(7, 4))
for (nome, v), cor in zip(sens.items(), ["#FFDF00", "#4FC3F7"]):
    plt.plot([c * 100 for c in custos], v, "o-", color=cor, lw=2, label=nome)
plt.axhline(0, color="gray", ls="--")
plt.xlabel("Custo por perna e por troca (%)")
plt.ylabel("Sharpe anualizado")
plt.legend()
plt.savefig("graficos_pairs_corrigido/grafico6_custos.png", dpi=200,
            bbox_inches="tight", facecolor="black")
plt.show()

#Passo 9: benchmarks e alfa contra o mercado
print(f"\nBenchmarks, comprar e segurar (retorno anual aritmético em excesso de RF = {RF_ANUAL:.1%}):")
for c in (A, B, MKT):
    r = np.nan_to_num(np.r_[np.nan, ret[c].values])[sel10] - RF_ANUAL / 252
    riq = np.cumprod(1 + r)
    print(f"  {c:<4} ret {r.mean() * 252:>6.1%} | vol {r.std(ddof=1) * np.sqrt(252):>6.1%} | "
          f"Sharpe {r.mean() / r.std(ddof=1) * np.sqrt(252):>5.2f} | queda máx. {(riq / np.maximum.accumulate(riq) - 1).min():>6.1%}")
y = r_ext[sel10]
X = sm.add_constant(np.nan_to_num(rM)[sel10])
m = sm.OLS(y, X).fit(cov_type="HAC", cov_kwds={"maxlags": 5})
print(f"Estratégia por extremos contra o mercado ({MKT}): beta {m.params[1]:.3f} (t {m.tvalues[1]:.2f}) | "
      f"alfa {m.params[0] * 252:.2%} ao ano (t {m.tvalues[0]:.2f})")
print("Lembrete: retorno e risco crescem juntos com a alavancagem; compare Sharpe, não só o retorno.")

#Passo 10: retorno acumulado na janela avaliada (comprar e segurar contra as regras)
janela = np.asarray(sel10)
for c in (A, B, MKT):
    acum = np.exp(np.nan_to_num(np.r_[np.nan, ret[c].values])[janela].sum()) - 1
    print(f"{c:<4} retorno acumulado: {acum:.1%}")
print(f"Regra por extremos: {np.prod(1 + r_ext[janela]) - 1:.1%} | incondicional: {np.prod(1 + r_inc[janela]) - 1:.1%}")